In [ ]:

dbutils.library.restartPython()
!pip install --upgrade pip
!pip install unidecode
!pip install rarfile
!pip install pdfplumber
!pip install nltk
!pip install openpyxl
!pip install xlrd
!pip install pytesseract
!pip install PyMuPDF
!pip install polars
%pip install dotenv

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, regexp_replace, regexp_extract
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp
from pyspark.sql import Row
import pandas as pd
import os
import requests
import json
import datetime
from suporte.support_functions import *

from config import get_config

In [ ]:
dbutils.widgets.text("projeto", "")
dbutils.widgets.text("fonte", "api")
projeto = dbutils.widgets.get("projeto").strip()
fonte = dbutils.widgets.get("fonte").strip().lower()
cfg = get_config(projeto)  # falha alto se `projeto` estiver vazio ou nao cadastrado

if fonte not in ("api", "vsol"):
    raise ValueError(f"Widget 'fonte' invalido: '{fonte}' (valores aceitos: api, vsol)")

In [ ]:
output_filename = dbutils.jobs.taskValues.get(
    taskKey="fetch_from_aga_api",
    key="output_filename",
    debugValue=None
)

if output_filename is None:
    raise RuntimeError("Missing output_filename from task_1")

print("Received filename:", output_filename)

In [ ]:
# output_filename = 'apiCAMPO_20260501_20260826_20260827_1257'

In [ ]:
project_name = projeto
project_path = "/mnt/wst/" + project_name
# Define paths
folder_bronze = project_path + "/BRONZE/"
folder_silver = project_path + "/SILVER/"
api_path = folder_silver + ("api_limpo/" if fonte == "api" else "api_limpo_vsol/")
api_original = folder_bronze + ("api/" if fonte == "api" else "vsol/") + output_filename

In [ ]:
# ------- SUMARIO DE EXECUCAO -------
execucao_steps = []

def log_step(etapa, df=None, status="Sucesso", observacoes="", registros_lidos=None, registros_escritos=None):
    n = df.count() if df is not None else 0
    execucao_steps.append({
        "etapa": etapa,
        "status": status,
        "registros_lidos": registros_lidos if registros_lidos is not None else n,
        "registros_escritos": registros_escritos if registros_escritos is not None else n,
        "flags": _collect_flags(df) if df is not None else "\u2014",
        "observacoes": observacoes,
    })

def _collect_flags(df, colunas=None):
    from pyspark.sql import functions as F
    flag_cols = colunas if colunas is not None else [c for c in df.columns if c.startswith("flag_")]
    triggered = []
    for col_name in flag_cols:
        count = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != "")).count()
        if count > 0:
            triggered.append(f"{col_name}({count})")
    return "; ".join(triggered) if triggered else "\u2014"

def persistir_log():
    from pyspark.sql import Row
    import datetime as _dt
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _execution_log_path = project_path + "/execution_log"
        _batch_ts = _dt.datetime.now().isoformat()
        _log_rows = [
            Row(
                batch_ts=_batch_ts,
                projeto=projeto,
                notebook=notebook_name.split("/")[-1],
                ordem=i,
                etapa=s["etapa"],
                status=s["status"],
                registros_lidos=int(s.get("registros_lidos") or 0),
                registros_escritos=int(s.get("registros_escritos") or 0),
                flags=str(s.get("flags") or "\u2014"),
                observacoes=str(s.get("observacoes") or ""),
                ts_registro=_dt.datetime.now().isoformat(),
            )
            for i, s in enumerate(execucao_steps)
        ]
        _log_df = spark.createDataFrame(_log_rows)
        _log_df.write.format("delta").mode("append").option("mergeSchema", "true").save(_execution_log_path)
        print(f"Log persistido: {len(_log_rows)} steps -> {_execution_log_path}")
    except Exception as _log_err:
        print(f"Aviso: falha ao persistir log de execucao \u2014 {_log_err}")

In [ ]:
try:
    print(f"📁 Reading API data from: {api_path}")
    df_api = spark.read.format("delta").load(api_path)
    print(f"📁 Reading original data from: {api_original}")
    df_original = spark.read.format("delta").load(api_original)

    execucao_steps.append({
        "etapa": "Leitura do DataFrame validado (Silver)",
        "status": "Sucesso",
        "registros_lidos": df_api.count(),
        "registros_escritos": df_api.count(),
        "flags": "\u2014",
        "observacoes": f"Dados lidos de {api_path}"
    })
except Exception as _e:
    execucao_steps.append({
        "etapa": "Leitura do DataFrame validado (Silver)", "status": "Erro",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014", "observacoes": str(_e)
    })
    persistir_log()
    raise

In [ ]:
from sharepoint_connector import download_file_by_name
import pandas as pd

# ===== DISPONIBILIDADE DO ESCOPO DE VALIDACAO =====
# O escopo contratado (planejamento de amostragem) vem de um Excel no SharePoint do projeto.
# Projetos que ainda nao tem esse arquivo publicado (ex.: CDM na largada) rodam o pipeline
# inteiro assim mesmo: a aba `api_validado` sai preenchida e as validacoes de escopo
# (metodo / holding time / parametro fora do escopo / faltantes por amostra) sao puladas.
# `escopo_disponivel` volta a ser True sozinho no dia em que o arquivo existir E a config
# (grupos_solo / grupos_asub) estiver preenchida -- sem mudar codigo.
escopo_configurado = bool(
    cfg.get("sharepoint_escopo_folder")
    and cfg.get("escopo_filename")
    and cfg.get("grupos_solo")
    and cfg.get("grupos_asub")
)
escopo_local_path = f"/tmp/{cfg.get('escopo_filename') or '_escopo_indisponivel.xlsx'}"
escopo_disponivel = False
df_escopo = None

if not escopo_configurado:
    log_step(
        "Download do escopo (SharePoint)",
        status="Aviso",
        registros_lidos=0, registros_escritos=0,
        observacoes=(
            f"Escopo de validacao nao configurado para o projeto '{projeto}' "
            "(escopo_filename / sharepoint_escopo_folder / grupos_solo / grupos_asub) -- "
            "validacoes de metodo / holding time / parametro fora do escopo puladas. "
            "`api_validado` segue normalmente."
        ),
    )
else:
    try:
        # ===== DOWNLOAD DO ESCOPO CONTRATADO DO SHAREPOINT =====
        result = download_file_by_name(
            folder_path=cfg["sharepoint_escopo_folder"],
            filename=cfg["escopo_filename"],
            local_path=escopo_local_path
        )
        if result["status"] != "success":
            raise Exception(f"Erro ao baixar escopo: {result['error']}")

        mapping_pd_1 = pd.read_excel(escopo_local_path, sheet_name=cfg["escopo_sheet_names"]["escopo"], dtype=str)
        df_escopo = spark.createDataFrame(mapping_pd_1)
        escopo_disponivel = True

        log_step(
            "Download do escopo (SharePoint)",
            df=df_escopo,
            observacoes=f"Escopo lido de {cfg['escopo_filename']} (aba {cfg['escopo_sheet_names']['escopo']})"
        )
    except Exception as _e:
        # Configurado, mas indisponivel nesta execucao (arquivo ainda nao publicado etc.).
        # Nao aborta o nb3 -- so pula as validacoes de escopo.
        escopo_disponivel = False
        log_step(
            "Download do escopo (SharePoint)",
            status="Aviso",
            registros_lidos=0, registros_escritos=0,
            observacoes=(
                f"Escopo configurado mas indisponivel ({_e}) -- validacoes de escopo "
                "puladas nesta execucao. `api_validado` segue normalmente."
            ),
        )


In [ ]:
from sharepoint_connector import download_file_by_name
import pandas as pd

df_planejamento = None

if escopo_disponivel:
    try:
        # ===== DOWNLOAD DO PLANEJAMENTO DE PARAMETROS POR ESTACAO - SOLO (SharePoint) =====
        result = download_file_by_name(
            folder_path=cfg["sharepoint_escopo_folder"],
            filename=cfg["escopo_filename"],
            local_path=escopo_local_path
        )
        if result["status"] != "success":
            raise Exception(f"Erro ao baixar planejamento: {result['error']}")

        planejamento_pd = pd.read_excel(
            escopo_local_path,
            sheet_name=cfg["escopo_sheet_names"]["solo"],
            dtype=str
        )
        df_planejamento = spark.createDataFrame(planejamento_pd)
        df_planejamento = df_planejamento.toDF(*[c.strip() for c in df_planejamento.columns])
        log_step(
            "Download do planejamento por estacao - Solo (SharePoint)",
            df=df_planejamento,
            observacoes=f"Planejamento lido de {cfg['escopo_filename']} (aba {cfg['escopo_sheet_names']['solo']})"
        )
    except Exception as _e:
        log_step(
            "Download do planejamento por estacao - Solo (SharePoint)",
            status="Erro", registros_lidos=0, registros_escritos=0, observacoes=str(_e)
        )
        persistir_log()
        raise


In [ ]:
from sharepoint_connector import download_file_by_name
import pandas as pd

df_planejamento_asub = None

if escopo_disponivel:
    try:
        # ===== DOWNLOAD DO PLANEJAMENTO DE PARAMETROS POR ESTACAO - AGUA SUBTERRANEA (SharePoint) =====
        result = download_file_by_name(
            folder_path=cfg["sharepoint_escopo_folder"],
            filename=cfg["escopo_filename"],
            local_path=escopo_local_path
        )
        if result["status"] != "success":
            raise Exception(f"Erro ao baixar planejamento (Agua Subterranea): {result['error']}")

        planejamento_asub_pd = pd.read_excel(
            escopo_local_path,
            sheet_name=cfg["escopo_sheet_names"]["asub"],
            dtype=str
        )
        df_planejamento_asub = spark.createDataFrame(planejamento_asub_pd)
        df_planejamento_asub = df_planejamento_asub.toDF(*[c.strip() for c in df_planejamento_asub.columns])
        # "Poco" vira "Sondagem" pra reaproveitar o mesmo nome de coluna/logica de ponto usado em Solo
        df_planejamento_asub = df_planejamento_asub.withColumnRenamed("Poço", "Sondagem")

        log_step(
            "Download do planejamento por estacao - Agua Subterranea (SharePoint)",
            df=df_planejamento_asub,
            observacoes=f"Planejamento lido de {cfg['escopo_filename']} (aba {cfg['escopo_sheet_names']['asub']})"
        )
    except Exception as _e:
        log_step(
            "Download do planejamento por estacao - Agua Subterranea (SharePoint)",
            status="Erro", registros_lidos=0, registros_escritos=0, observacoes=str(_e)
        )
        persistir_log()
        raise


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

if escopo_disponivel:
    # --- SOLO ---
    grupos_cols_solo = cfg["grupos_solo"]

    stack_expr_solo = "stack({0}, {1}) as (Planejamento, marcado)".format(
        len(grupos_cols_solo),
        ", ".join([f"'{c}', `{c}`" for c in grupos_cols_solo])
    )

    df_planejamento_long_solo = (
        df_planejamento
        .select(
            F.col("Sondagem"),
            F.col("Total de Amostras").cast("int").alias("totalAmostrasPlanejado"),
            F.expr(stack_expr_solo)
        )
        .filter(F.col("marcado").isNotNull())
        .drop("marcado")
        .withColumn("Matriz", F.lit("Solo"))
    )

    # --- AGUA SUBTERRANEA ---
    # Os nomes de coluna da aba ASUB nao batem 1:1 com os valores de "Planejamento" do
    # EscopoCampoAnalises (ex.: "TPH FP" na planilha de pontos vs "TPH" no escopo; "Coliformes" vs
    # "Coliformes Totais"), por isso existe esse de-para explicito por projeto, em
    # cfg["mapa_grupo_asub"]. Ajustar em config/projetos.py sempre que o nome de algum grupo mudar em
    # qualquer uma das duas abas daquele projeto.
    grupos_cols_asub = cfg["grupos_asub"]
    mapa_grupo_asub = cfg["mapa_grupo_asub"]

    stack_expr_asub = "stack({0}, {1}) as (PlanejamentoASUB, marcado)".format(
        len(grupos_cols_asub),
        ", ".join([f"'{c}', `{c}`" for c in grupos_cols_asub])
    )

    mapa_expr_asub = F.create_map([F.lit(x) for par in mapa_grupo_asub.items() for x in par])

    df_planejamento_long_asub = (
        df_planejamento_asub
        .select(
            F.col("Sondagem"),
            F.col("Total de Amostras").cast("int").alias("totalAmostrasPlanejado"),
            F.expr(stack_expr_asub)
        )
        .filter(F.col("marcado").isNotNull())
        .drop("marcado")
        .withColumn("Planejamento", mapa_expr_asub[F.col("PlanejamentoASUB")])
        .drop("PlanejamentoASUB")
        .withColumn("Matriz", F.lit("Agua Subterranea"))
    )

    # --- Combinado: Solo + Agua Subterranea, com a coluna Matriz identificando cada um ---
    df_planejamento_long = df_planejamento_long_solo.unionByName(df_planejamento_long_asub)
    df_planejamento_long = df_planejamento_long.withColumnRenamed("Sondagem", "Station")
else:
    # Sem escopo: planejamento vazio -- os joins a jusante nao marcam nada como planejado
    _schema_planejamento_long = StructType([
        StructField("Station", StringType(), True),
        StructField("totalAmostrasPlanejado", IntegerType(), True),
        StructField("Planejamento", StringType(), True),
        StructField("Matriz", StringType(), True),
    ])
    df_planejamento_long = spark.createDataFrame([], _schema_planejamento_long)

df_planejamento_long.display()


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

if escopo_disponivel:
    df_escopo_sel = df_escopo.select(
            F.col("Parametros HGA"),
            F.col("Matriz").alias("matriz_escopo"),
            F.col("Planejamento"),
            F.col("Método"),
            F.col("Holding Time"))
else:
    _schema_escopo_sel = StructType([
        StructField("Parametros HGA", StringType(), True),
        StructField("matriz_escopo", StringType(), True),
        StructField("Planejamento", StringType(), True),
        StructField("Método", StringType(), True),
        StructField("Holding Time", StringType(), True),
    ])
    df_escopo_sel = spark.createDataFrame([], _schema_escopo_sel)

df_escopo_sel.display()


In [ ]:
df_planejamento_long.display()

In [ ]:
df_parametros_planejados = (
    df_planejamento_long
    .join(
        df_escopo_sel
            .select(
                col("Planejamento"),
                col("matriz_escopo").alias("Matriz"),
                col("Parametros HGA").alias("Parametro"),
                col("Método"),
                col("Holding Time"),
            ),
        on=["Planejamento", "Matriz"],
        how="left",
    )
).select('Station', 'Matriz', 'totalAmostrasPlanejado', 'Planejamento', 'Parametro')

In [ ]:
df_parametros_planejados.display()

In [ ]:
# Aqui vou filtrar o df para pegar somente onde o df_api deu erro de mapeamento, nao quero que entra nas analises
df_parametro_errado = df_api.filter(F.col("flag_padronizacao") == "Parametro nao mapeado")
df_api = df_api.filter(
    F.col("flag_padronizacao").isNull() | (F.col("flag_padronizacao") != "Parametro nao mapeado")
)

In [ ]:
df_api.display()

In [ ]:
from pyspark.sql import functions as F

try:
    if escopo_disponivel:
        # Recria o recorte direto de df_escopo (garante os nomes/tipos originais das colunas)
        df_escopo_sel = df_escopo.select(
            F.col("Parametros HGA"),
            F.col("Matriz").alias("matriz_escopo"),
            F.col("Planejamento"),
            F.col("Método"),
            F.col("Holding Time")
        )
    # Sem escopo: mantem o df_escopo_sel vazio criado na celula anterior. O join left
    # abaixo preserva todas as linhas de df_api e deixa "Método"/"Holding Time" nulos.

    df_api = df_api.join(
        df_escopo_sel,
        (df_api.parametroCorrigido == df_escopo_sel["Parametros HGA"]) &
        (df_api.matriz == df_escopo_sel.matriz_escopo),
        "left"
    ).drop("Parametros HGA", "matriz_escopo", 'Planejamento')

    log_step(
        "Cruzamento com o escopo",
        df=df_api,
        observacoes=(
            "Join left de df_api com o escopo (chave: ShortName x Parametros HGA, matriz x Matriz)"
            if escopo_disponivel
            else "Escopo indisponivel -- colunas Método/Holding Time criadas nulas, sem cruzamento"
        )
    )
except Exception as _e:
    log_step("Cruzamento com o escopo", status="Erro", registros_lidos=0, registros_escritos=0, observacoes=str(_e))
    persistir_log()
    raise


# Parametro Extra, Método Diferente, HoldingTime

In [ ]:
from pyspark.sql import functions as F

if not escopo_disponivel:
    # Sem escopo contratado nao ha como dizer o que esta "fora do escopo" -- flag fica nula
    df_api = df_api.withColumn("flag_parametro_fora_escopo", F.lit(None).cast("string"))
    execucao_steps.append({
        "etapa": "Validacao de parametros fora do escopo",
        "status": "Aviso",
        "registros_lidos": df_api.count(),
        "registros_escritos": 0,
        "flags": "—",
        "observacoes": "Escopo indisponivel -- validacao de parametro fora do escopo pulada"
    })
else:
    try:
        parametros_no_escopo = (
            df_escopo_sel
            .select(F.col("Parametros HGA").alias("param_no_escopo"))
            .distinct()
        )

        df_api = df_api.join(
            parametros_no_escopo,
            df_api["parametroCorrigido"] == parametros_no_escopo["param_no_escopo"],
            "left"
        ).withColumn(
            "flag_parametro_fora_escopo",
            F.when(
                F.col("flag_station_sem_mapeamento").isNotNull(),
                F.lit(None).cast("string")
            ).when(
                F.col("param_no_escopo").isNull(),
                F.lit("Parametro fora do escopo contratado")
            ).otherwise(F.lit(None).cast("string"))
        ).drop("param_no_escopo")

        qtd_parametro_extra = df_api.filter(F.col("flag_parametro_fora_escopo").isNotNull()).count()

        execucao_steps.append({
            "etapa": "Validacao de parametros fora do escopo",
            "status": "Aviso" if qtd_parametro_extra > 0 else "Sucesso",
            "registros_lidos": df_api.count(),
            "registros_escritos": df_api.count(),
            "flags": _collect_flags(df_api, ["flag_parametro_fora_escopo"]),
            "observacoes": f"{qtd_parametro_extra} parametro(s) fora do escopo contratado"
        })
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Validacao de parametros fora do escopo", "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "—", "observacoes": str(_e)
        })
        persistir_log()
        raise


In [ ]:
from pyspark.sql import functions as F

if not escopo_disponivel:
    df_api = df_api.withColumn("flag_metodo", F.lit(None).cast("string"))
    execucao_steps.append({
        "etapa": "Validacao de metodo analitico",
        "status": "Aviso",
        "registros_lidos": df_api.count(),
        "registros_escritos": 0,
        "flags": "—",
        "observacoes": "Escopo indisponivel -- validacao de metodo analitico pulada"
    })
else:
    try:
        df_api = df_api.withColumn(
            "flag_metodo",
            F.when(
                F.col("flag_station_sem_mapeamento").isNotNull(),
                F.lit(None).cast("string")
            ).when(
                F.col("flag_parametro_fora_escopo").isNotNull(),
                F.lit(None).cast("string")
            ).when(
                F.col("metodoAnalise") != F.col("Método"),
                F.lit("Metodo diferente do escopo")
            ).otherwise(F.lit(None).cast("string"))
        )

        qtd_metodo_diferente = df_api.filter(F.col("flag_metodo").isNotNull()).count()

        execucao_steps.append({
            "etapa": "Validacao de metodo analitico",
            "status": "Aviso" if qtd_metodo_diferente > 0 else "Sucesso",
            "registros_lidos": df_api.count(),
            "registros_escritos": df_api.count(),
            "flags": _collect_flags(df_api, ["flag_metodo"]),
            "observacoes": f"{qtd_metodo_diferente} parametro(s) com metodo analitico divergente do escopo"
        })
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Validacao de metodo analitico", "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "—", "observacoes": str(_e)
        })
        persistir_log()
        raise


In [ ]:
from pyspark.sql import functions as F

if not escopo_disponivel:
    # diasDecorridos nao depende do escopo -- segue sendo calculado (vai pro Excel como
    # holding_time_executado). Sem escopo nao ha "Holding Time (dias)" planejado, entao
    # a flag fica nula.
    df_api = (
        df_api
        .withColumn("Holding Time (dias)", F.lit(None).cast("int"))
        .withColumn(
            "diasDecorridos",
            F.datediff(F.col("dataRecebLab_valida"), F.col("dataHoraAmostragem_valida"))
        )
        .withColumn("flag_holding_time", F.lit(None).cast("string"))
    )
    execucao_steps.append({
        "etapa": "Validacao de holding time",
        "status": "Aviso",
        "registros_lidos": df_api.count(),
        "registros_escritos": 0,
        "flags": "—",
        "observacoes": "Escopo indisponivel -- validacao de holding time pulada (diasDecorridos calculado mesmo assim)"
    })
else:
    try:
        df_api = df_api.withColumn(
            "Holding Time (dias)",
            F.regexp_extract(F.col("Holding Time"), r"(\d+)", 1).cast("int")
        ).withColumn(
            "diasDecorridos",
            F.datediff(F.col("dataRecebLab_valida"), F.col("dataHoraAmostragem_valida"))
        ).withColumn(
            "flag_holding_time",
            F.when(
                F.col("flag_station_sem_mapeamento").isNotNull(),
                F.lit(None).cast("string")
            ).when(
                F.col("flag_parametro_fora_escopo").isNotNull(),
                F.lit(None).cast("string")
            ).when(
                F.col("Holding Time (dias)").isNotNull() & (F.col("diasDecorridos") > F.col("Holding Time (dias)")),
                F.lit("Holding time fora do prazo")
            ).otherwise(F.lit(None).cast("string"))
        )

        qtd_holding_fora_prazo = df_api.filter(F.col("flag_holding_time").isNotNull()).count()

        execucao_steps.append({
            "etapa": "Validacao de holding time",
            "status": "Aviso" if qtd_holding_fora_prazo > 0 else "Sucesso",
            "registros_lidos": df_api.count(),
            "registros_escritos": df_api.count(),
            "flags": _collect_flags(df_api, ["flag_holding_time"]),
            "observacoes": f"{qtd_holding_fora_prazo} parametro(s) com holding time fora do prazo"
        })
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Validacao de holding time", "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "—", "observacoes": str(_e)
        })
        persistir_log()
        raise


### Validacoes Por Station

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

if not escopo_disponivel:
    # Sem planejamento por estacao nao da pra apontar faltantes/extras por amostra
    _schema_val_amostra = StructType([
        StructField("Station", StringType(), True),
        StructField("Matriz", StringType(), True),
        StructField("Amostra", StringType(), True),
        StructField("Parametro", StringType(), True),
    ])
    df_parametros_faltantes_por_amostra = spark.createDataFrame([], _schema_val_amostra)
    df_parametros_extras_de_grupo_por_amostra = spark.createDataFrame([], _schema_val_amostra)
    execucao_steps.append({
        "etapa": "Validacao de parametros faltantes/extras por amostra",
        "status": "Aviso",
        "registros_lidos": 0,
        "registros_escritos": 0,
        "flags": "—",
        "observacoes": "Escopo indisponivel -- validacao de faltantes/extras por amostra pulada"
    })
else:
    try:
        # 1. Todas as amostras reais que existem (uma linha por ponto de coleta)
        df_amostras_reais = (
            df_api
            .filter(F.col("flag_station_sem_mapeamento").isNull())
            .select(
                F.col("alternate_name").alias("Station"),
                F.col("matriz").alias("Matriz"),
                F.col("descricaoAmostra_final").alias("Amostra"),
            )
            .distinct()
        )

        # 2. O que CADA amostra deveria ter: todo parametro planejado pra aquele Station+Matriz
        df_esperado_por_amostra = (
            df_amostras_reais
            .join(
                df_parametros_planejados.select("Station", "Matriz", "Parametro"),
                on=["Station", "Matriz"],
                how="inner"
            )
        )

        # 3. O que REALMENTE foi lancado (exclui station sem mapeamento e parametros fora do escopo cadastrado)
        df_lancado_por_amostra = (
            df_api
            .filter(
                F.col("flag_station_sem_mapeamento").isNull() &
                F.col("flag_parametro_fora_escopo").isNull()
            )
            .select(
                F.col("alternate_name").alias("Station"),
                F.col("matriz").alias("Matriz"),
                F.col("descricaoAmostra_final").alias("Amostra"),
                F.col("parametroCorrigido").alias("Parametro"),
            )
            .distinct()
        )

        # 4. FALTANTES: esperado que nao tem correspondencia no lancado
        df_parametros_faltantes_por_amostra = (
            df_esperado_por_amostra
            .join(df_lancado_por_amostra, on=["Station", "Matriz", "Amostra", "Parametro"], how="left_anti")
        )

        # 5. EXTRAS DE GRUPO: lancado que nao esta no esperado daquela station/matriz
        #    (existe no escopo geral, mas nao e do grupo planejado pra esse ponto)
        df_parametros_extras_de_grupo_por_amostra = (
            df_lancado_por_amostra
            .join(df_esperado_por_amostra, on=["Station", "Matriz", "Amostra", "Parametro"], how="left_anti")
        )

        qtd_faltantes = df_parametros_faltantes_por_amostra.count()
        qtd_extras_grupo = df_parametros_extras_de_grupo_por_amostra.count()

        execucao_steps.append({
            "etapa": "Validacao de parametros faltantes/extras por amostra",
            "status": "Aviso" if (qtd_faltantes > 0 or qtd_extras_grupo > 0) else "Sucesso",
            "registros_lidos": df_lancado_por_amostra.count(),
            "registros_escritos": qtd_faltantes + qtd_extras_grupo,
            "flags": f"faltantes({qtd_faltantes}); extras_de_grupo({qtd_extras_grupo})",
            "observacoes": f"{qtd_faltantes} parametro(s) planejado(s) nao lancado(s) por amostra; {qtd_extras_grupo} parametro(s) lancado(s) fora do grupo planejado por amostra"
        })
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Validacao de parametros faltantes/extras por amostra", "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "—", "observacoes": str(_e)
        })
        persistir_log()


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

if not escopo_disponivel:
    # Aba "resumo_amostra" sai vazia nesta execucao (schema preservado pro export do Excel)
    _schema_resumo_amostra = StructType([
        StructField("Station", StringType(), True),
        StructField("Matriz", StringType(), True),
        StructField("Amostra", StringType(), True),
        StructField("parametros_extras", StringType(), True),
        StructField("parametros_faltantes", StringType(), True),
        StructField("parametros_fora_do_escopo", StringType(), True),
        StructField("qtd_parametros_total", IntegerType(), True),
        StructField("qtd_extras", IntegerType(), True),
        StructField("qtd_faltantes", IntegerType(), True),
        StructField("qtd_fora_do_escopo", IntegerType(), True),
    ])
    df_resumo_amostra = spark.createDataFrame([], _schema_resumo_amostra)
    log_step(
        "Resumo de parametros faltantes/extras por amostra",
        df=df_resumo_amostra,
        status="Aviso",
        observacoes="Escopo indisponivel -- resumo por amostra vazio nesta execucao"
    )
else:
    try:
        # Parametros que existem no grupo planejado mas nao foram lancados
        df_extras_agrupado = (
            df_parametros_extras_de_grupo_por_amostra
            .groupBy("Station", "Matriz", "Amostra")
            .agg(F.concat_ws("; ", F.collect_set("Parametro")).alias("parametros_extras"))
        )

        df_faltantes_agrupado = (
            df_parametros_faltantes_por_amostra
            .groupBy("Station", "Matriz", "Amostra")
            .agg(F.concat_ws("; ", F.collect_set("Parametro")).alias("parametros_faltantes"))
        )

        # Parametros que nem existem no df_escopo (flag_parametro_fora_escopo), por amostra
        df_extra_escopo_agrupado = (
            df_api
            .filter(
                F.col("flag_station_sem_mapeamento").isNull() &
                F.col("flag_parametro_fora_escopo").isNotNull()
            )
            .select(
                F.col("alternate_name").alias("Station"),
                F.col("matriz").alias("Matriz"),
                F.col("descricaoAmostra_final").alias("Amostra"),
                F.col("parametroCorrigido").alias("Parametro"),
            )
            .distinct()
            .groupBy("Station", "Matriz", "Amostra")
            .agg(F.concat_ws("; ", F.collect_set("Parametro")).alias("parametros_fora_do_escopo"))
        )

        # Total de parametros distintos lancados por amostra (todos, independente de flag)
        df_total_parametros_agrupado = (
            df_api
            .filter(F.col("flag_station_sem_mapeamento").isNull())
            .select(
                F.col("alternate_name").alias("Station"),
                F.col("matriz").alias("Matriz"),
                F.col('idAmostra'),
                F.col("descricaoAmostra_final").alias("Amostra"),
                F.col("parametroCorrigido").alias("Parametro"),
            )
            .distinct()
            .groupBy("Station", "Matriz", "Amostra")
            .agg(F.countDistinct("Parametro").alias("qtd_parametros_total"))
        )

        df_resumo_amostra = (
            df_extras_agrupado
            .join(df_faltantes_agrupado, on=["Station", "Matriz", "Amostra"], how="full")
            .join(df_extra_escopo_agrupado, on=["Station", "Matriz", "Amostra"], how="full")
            .join(df_total_parametros_agrupado, on=["Station", "Matriz", "Amostra"], how="full")
            .withColumn("qtd_extras", F.when(F.col("parametros_extras").isNotNull(), F.size(F.split(F.col("parametros_extras"), "; "))).otherwise(F.lit(0)))
            .withColumn("qtd_faltantes", F.when(F.col("parametros_faltantes").isNotNull(), F.size(F.split(F.col("parametros_faltantes"), "; "))).otherwise(F.lit(0)))
            .withColumn("qtd_fora_do_escopo", F.when(F.col("parametros_fora_do_escopo").isNotNull(), F.size(F.split(F.col("parametros_fora_do_escopo"), "; "))).otherwise(F.lit(0)))
            .withColumn("qtd_parametros_total", F.coalesce(F.col("qtd_parametros_total"), F.lit(0)))
            .orderBy("Station", "Matriz", "Amostra")
        )

        log_step(
            "Resumo de parametros faltantes/extras por amostra",
            df=df_resumo_amostra,
            observacoes="Consolidado por Station+Matriz+Amostra: parametros fora do grupo planejado, planejados nao lancados, fora do escopo cadastrado e total lancado"
        )
    except Exception as _e:
        log_step("Resumo de parametros faltantes/extras por amostra", status="Erro", registros_lidos=0, registros_escritos=0, observacoes=str(_e))
        persistir_log()
        raise

df_resumo_amostra.display()


# Montagem Excel

In [ ]:
df_api.display()

In [ ]:
from pyspark.sql.functions import col
col_map_1 = {
    'idAmostra': 'sample_id',
    'descricaoAmostra_final': 'sample_name',
    'dataLiberacao_valida': "DataLiberacao",
    'dataHoraAmostragem_valida': 'sample_date',
    'dataRecebLab_valida': 'sample_date_received',
    'dataEnvioLab_valida': 'sample_date_sent',
    'tipoAnalise': 'sample_type',
    'dataHoraAnalise_valida': 'AnalysisDate',
    'codigoHga': 'Station',
    'matriz': 'samp_matrix',
    'laboratorio': 'laboratory',
    'codigoQualidade': 'quality_code',
    'frequencia': 'frequency',
    'OrigemArquivo': 'data_source',
    'parametroCorrigido': 'chemical_name',
    'resultadoCorrigido': 'value',
    'metodoAnalise': 'analysis_method',
    'parametroOriginal': 'original_parameter',
    'resultadoOriginal': 'original_result',
    'unidadeOriginal': 'original_unit',
    'limiteQuantificacao': 'quantification_limit',
    'resultadoTexto': 'text_result',
    'unidade_para': 'unit',
    'flag_conversao': 'comment',
    'Método': 'metodo_planejado',
    'Holding Time': 'hodlgin_time_planejado',
    'diasDecorridos': "holding_time_executado",
    'comentario_parametro': 'result_comment',
    'comentario_amostra': 'sample_comment'
    }

colunas_finais_1 = [
   'alternate_name', 'Station', 'sample_name', 'sample_date', 'sample_date_received', 'sample_date_sent', "DataLiberacao", 'AnalysisDate', 'laboratory', 'quality_code', 'frequency', 'quantification_limit', 'sample_comment', 'samp_matrix', 'chemical_name', 'unit', 'value', 'qualifier', 'sample_id', 'analysis_method', 'original_parameter', 'original_unit', 'original_result', 'text_result', 'comment', 'result_comment', 'flag_parametro_fora_escopo', 'flag_metodo', 'flag_holding_time', 'metodo_planejado', 'hodlgin_time_planejado', "holding_time_executado", 'flag_station_sem_mapeamento', 'flag_padronizacao'
]
select_exprs_1 = []
for nome_col in colunas_finais_1:
    original = next((k for k, v in col_map_1.items() if v == nome_col), None)
    if original and original in df_api.columns:
        select_exprs_1.append(F.col(original).alias(nome_col))
    elif nome_col in df_api.columns and nome_col not in col_map_1:
        select_exprs_1.append(F.col(nome_col))
    else:
        select_exprs_1.append(F.lit("").alias(nome_col))

df_formatado_1 = df_api.select(select_exprs_1)
condicao_com_flag = col('flag_station_sem_mapeamento').isNotNull() | col('flag_padronizacao').isNotNull()
df_formatado_final = df_formatado_1.filter(~condicao_com_flag)

# Separa linhas sem Station (sem match) do restante
df_sem_match = df_formatado_1.filter(col('flag_station_sem_mapeamento').isNotNull())

# ------- Registro: amostras distintas no resultado final -------
qtd_amostras_distintas = df_formatado_1.select("sample_name").distinct().count()
execucao_steps.append({
    "etapa": "Amostras distintas (sample_name)",
    "status": "Sucesso",
    "registros_lidos": df_formatado_1.count(),
    "registros_escritos": qtd_amostras_distintas,
    "flags": "\u2014",
    "observacoes": f"{qtd_amostras_distintas} amostras distintas (sample_name) no resultado final"
})


In [ ]:
col_map_2 = dict(col_map_1)

colunas_finais_2 = [
   'alternate_name', 'Station', 'sample_name', 'sample_date', 'sample_date_received', 'sample_date_sent', "DataLiberacao", 'AnalysisDate', 'laboratory', 'quality_code', 'frequency', 'quantification_limit', 'sample_comment', 'samp_matrix', 'chemical_name', 'unit', 'value', 'qualifier', 'sample_id', 'analysis_method', 'original_parameter', 'original_unit', 'original_result', 'text_result', 'comment', 'result_comment', 'flag_parametro_extra', 'flag_metodo', 'flag_holding_time', 'metodo_planejado', 'hodlgin_time_planejado', "holding_time_executado"
]
select_exprs_2 = []
for nome_col in colunas_finais_2:
    original = next((k for k, v in col_map_2.items() if v == nome_col), None)
    if original and original in df_parametro_errado.columns:
        select_exprs_2.append(F.col(original).alias(nome_col))
    elif nome_col in df_parametro_errado.columns and nome_col not in col_map_1:
        select_exprs_2.append(F.col(nome_col))
    else:
        select_exprs_2.append(F.lit("").alias(nome_col))

df_formatado_2 = df_parametro_errado.select(select_exprs_2)

In [ ]:
# --- Etapa 3: Cria um DataFrame para cada flag preenchida
# (mesma estrutura de df_formatado_1, so filtrando as linhas)

df_parametro_fora_escopo = df_formatado_1.filter(
    F.col('flag_parametro_fora_escopo').isNotNull() & (F.trim(F.col('flag_parametro_fora_escopo')) != '')
)
df_metodo = df_formatado_1.filter(
    F.col('flag_metodo').isNotNull() & (F.trim(F.col('flag_metodo')) != '')
)
df_holding_time = df_formatado_1.filter(
    F.col('flag_holding_time').isNotNull() & (F.trim(F.col('flag_holding_time')) != '')
)

In [ ]:
# =========================
# Column sanitization
# =========================
from pyspark.sql import DataFrame
import re

def sanitize_column_names(columns):
    sanitized = []
    for col in columns:
        new_col = re.sub(r"[ ,;{}()\n\t=]", "§", col)
        sanitized.append(new_col)
    return sanitized

def sanitize_spark_columns(df):
    new_cols = sanitize_column_names(df.columns)
    return df.toDF(*new_cols)


In [ ]:
# =========================
# Save helper -- com log de execucao
# =========================
def save_spark_if_not_empty(df: DataFrame, path, etapa):
    try:
        if not df.rdd.isEmpty():
            df_s = sanitize_spark_columns(df)
            n = df_s.count()
            (
                df_s.write
                .format("delta")
                .mode("overwrite")
                .option('overwriteSchema', 'true')
                .save(path)
            )
            execucao_steps.append({
                "etapa": etapa,
                "status": "Sucesso",
                "registros_lidos": n,
                "registros_escritos": n,
                "flags": _collect_flags(df_s),
                "observacoes": f"Gravado em {path}"
            })
        else:
            execucao_steps.append({
                "etapa": etapa,
                "status": "Aviso",
                "registros_lidos": 0,
                "registros_escritos": 0,
                "flags": "\u2014",
                "observacoes": f"DataFrame vazio \u2014 nada gravado em {path}"
            })
    except Exception as _e:
        execucao_steps.append({
            "etapa": etapa, "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014",
            "observacoes": str(_e)
        })
        persistir_log()
        raise


# =========================
# Save paths
# =========================
folder_silver_validado = project_path + "/SILVER/api_validado"
folder_silver_resumo_estacao = project_path + "/SILVER/resumo_por_estacao"
folder_silver_resumo_amostra = project_path + "/SILVER/resumo_amostra"
folder_silver_extra    = project_path + "/SILVER/parametro_extra"
folder_silver_metodo   = project_path + "/SILVER/metodo"
folder_silver_holding  = project_path + "/SILVER/holding"
folder_silver_sample_name = project_path + '/SILVER/erro_sampleName'
folder_silver_erro_mapeamento = project_path + '/SILVER/erro_mapeamento'


save_path_validado = f"{folder_silver_validado}/{output_filename}"
save_path_resumo_amostra = f"{folder_silver_resumo_amostra}/{output_filename}"
save_path_extra    = f"{folder_silver_extra}/{output_filename}"
save_path_metodo   = f"{folder_silver_metodo}/{output_filename}"
save_path_holding  = f"{folder_silver_holding}/{output_filename}"
save_path_sample_name = f"{folder_silver_sample_name}/{output_filename}"
save_path_erro_mapeamento = f'{folder_silver_erro_mapeamento}/{output_filename}'


# =========================
# Save -- tudo Spark
# =========================
save_spark_if_not_empty(df_formatado_final,   save_path_validado,       "Gravacao Silver - API Validado")
save_spark_if_not_empty(df_resumo_amostra,    save_path_resumo_amostra, "Gravacao Silver - Resumo Amostra")
save_spark_if_not_empty(df_parametro_fora_escopo, save_path_extra,      "Gravacao Silver - Parametro Extra")
save_spark_if_not_empty(df_metodo,            save_path_metodo,         "Gravacao Silver - Metodo")
save_spark_if_not_empty(df_holding_time,      save_path_holding,        "Gravacao Silver - Holding Time")
save_spark_if_not_empty(df_formatado_2,       save_path_erro_mapeamento, "Gravacao Silver - Erro de Mapeamento")
save_spark_if_not_empty(df_sem_match,         save_path_sample_name,    "Gravacao Silver - Erro de Mapeamento Sample_Name")


In [ ]:
# =========================
# Export consolidado em Excel (para o notebook de envio ao SharePoint)
# =========================
import pandas as pd
import datetime as _dt
import os

def export_excel_multisheet(sheets: dict, folder_path: str, etapa: str):
    try:
        non_empty = {name: df for name, df in sheets.items() if not df.rdd.isEmpty()}

        if not non_empty:
            execucao_steps.append({
                "etapa": etapa,
                "status": "Aviso",
                "registros_lidos": 0,
                "registros_escritos": 0,
                "flags": "\u2014",
                "observacoes": "Todos os DataFrames vazios \u2014 Excel nao gerado"
            })
            return None

        data_str = _dt.datetime.now().strftime("%Y%m%d")
        filename = f"export_df_{data_str}.xlsx"
        dbfs_path = f"{folder_path}/{filename}"
        local_tmp_path = f"/tmp/{filename}"
        local_tmp_crc = f"/tmp/.{filename}.crc"

        for _p in (local_tmp_path, local_tmp_crc):
            if os.path.exists(_p):
                os.remove(_p)

        total_linhas = 0
        with pd.ExcelWriter(local_tmp_path, engine="openpyxl") as writer:
            for sheet_name, df in non_empty.items():
                pdf = df.toPandas()
                pdf.to_excel(writer, sheet_name=sheet_name[:31], index=False)
                total_linhas += len(pdf)

        dbutils.fs.mkdirs(folder_path)
        dbutils.fs.cp(f"file:{local_tmp_path}", dbfs_path, True)
        os.remove(local_tmp_path)
        if os.path.exists(local_tmp_crc):
            os.remove(local_tmp_crc)

        execucao_steps.append({
            "etapa": etapa,
            "status": "Sucesso",
            "registros_lidos": total_linhas,
            "registros_escritos": total_linhas,
            "flags": "\u2014",
            "observacoes": f"Excel gravado em {dbfs_path} \u2014 abas: {', '.join(non_empty.keys())}"
        })
        return dbfs_path
    except Exception as _e:
        execucao_steps.append({
            "etapa": etapa, "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014",
            "observacoes": str(_e)
        })
        persistir_log()
        raise


folder_silver_export = project_path + "/SILVER/export_continuo"

sheets = {
    "api_validado":    df_formatado_final,
    'df_original': df_original,
    'Parametros sem Mapeamento': df_formatado_2,
    'Statinos Sem Mapeamento': df_sem_match,
    "resumo_amostra":  df_resumo_amostra,
    "parametro_fora_escopo": df_parametro_fora_escopo,
    "metodo":          df_metodo,
    "holding":         df_holding_time,
}

export_excel_multisheet(sheets, folder_silver_export, "Exportacao Excel - Consolidado SharePoint")

In [ ]:
# ------- PERSISTIR LOG DE EXECUCAO -------
persistir_log()
display(pd.DataFrame(execucao_steps))